In [1]:
import math
from pathlib import Path
from typing import List, Dict

import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


# ==========================
# Configurações e Constantes
# ==========================

DATASET_PATH = Path(
    r"C:/Users/bianc/OneDrive/Documents/1-Estudos/2-DNC/1-Data Science/"
    r"1-Material/Materia7-EstatisticaComPython/IA/IA em análise de Dados/Salary_dataset.csv"
)

TARGET_COLUMN = "Salary"
FEATURE_COLUMN = "YearsExperience"
COLUMNS_FOR_IQR = [FEATURE_COLUMN, TARGET_COLUMN]


# ==========================
# Funções Utilitárias
# ==========================

def load_salary_dataset(path: Path) -> pd.DataFrame:
    """
    Carrega o dataset de salários a partir de um arquivo CSV
    e remove colunas de índice vazando (ex.: 'Unnamed: 0').

    Parameters
    ----------
    path : Path
        Caminho completo para o arquivo CSV.

    Returns
    -------
    pd.DataFrame
        DataFrame com os dados prontos para análise.
    """
    df = pd.read_csv(path)

    # Remove colunas de índice automático salvas no CSV (caso existam)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    return df


def detect_outliers_iqr(df: pd.DataFrame, columns: List[str], iqr_multiplier: float = 1.5) -> Dict[str, pd.DataFrame]:
    """
    Detecta outliers usando o método do IQR (Interquartile Range) para
    um conjunto de colunas numéricas.

    Para cada coluna:
        - Calcula Q1 (25%), Q3 (75%) e IQR = Q3 - Q1
        - Define limites: [Q1 - k*IQR, Q3 + k*IQR], onde k = iqr_multiplier
        - Filtra linhas com valores fora desses limites.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame com os dados.
    columns : List[str]
        Lista de nomes das colunas numéricas a serem analisadas.
    iqr_multiplier : float, opcional (default=1.5)
        Fator multiplicador do IQR para definir os limites.

    Returns
    -------
    Dict[str, pd.DataFrame]
        Dicionário onde:
            - chave  : nome da coluna
            - valor  : DataFrame contendo apenas as linhas outliers daquela coluna
    """
    outliers_por_coluna: Dict[str, pd.DataFrame] = {}

    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - iqr_multiplier * iqr
        upper_bound = q3 + iqr_multiplier * iqr

        mask_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers_col = df[mask_outliers]

        outliers_por_coluna[col] = outliers_col

        print(f"\nColuna: {col}")
        print(f"Limites IQR ({iqr_multiplier}x): [{lower_bound:.2f}, {upper_bound:.2f}]")
        print("Outliers encontrados:")
        print(outliers_col if not outliers_col.empty else "Nenhum")

    return outliers_por_coluna


def analyze_regression_influence(df: pd.DataFrame, feature_col: str, target_col: str, threshold: float = 2.0) -> pd.DataFrame:
    """
    Ajusta um modelo de regressão linear via statsmodels OLS (com intercepto)
    e calcula métricas de influência para detectar pontos potencialmente
    problemáticos (outliers de regressão).

    Métricas adicionadas ao DataFrame:
        - resid_studentized : resíduos studentizados internos
        - cooks_distance    : distância de Cook

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame com os dados.
    feature_col : str
        Nome da coluna de feature (X).
    target_col : str
        Nome da coluna target (y).
    threshold : float, opcional (default=2.0)
        Limite absoluto para o resíduo studentizado. Pontos com |resíduo| > threshold
        serão considerados outliers de regressão.

    Returns
    -------
    pd.DataFrame
        Subconjunto do DataFrame contendo apenas os pontos marcados como outliers
        pelo critério de resíduo studentizado.
    """
    x = df[[feature_col]]
    y = df[target_col]

    # Adiciona constante para estimar intercepto (β0)
    x_with_const = sm.add_constant(x)

    ols_model = sm.OLS(y, x_with_const).fit()
    influence = ols_model.get_influence()

    df["resid_studentized"] = influence.resid_studentized_internal
    df["cooks_distance"] = influence.cooks_distance[0]

    outliers_reg = df[df["resid_studentized"].abs() > threshold]

    print("\n=== Outliers de Regressão (statsmodels OLS) ===")
    print(f"Total de pontos com |resíduo studentizado| > {threshold}: {len(outliers_reg)}")
    if not outliers_reg.empty:
        print(outliers_reg[[feature_col, target_col, "resid_studentized", "cooks_distance"]])
    else:
        print("Nenhum ponto identificado como outlier de regressão.")

    return outliers_reg


def train_linear_regression(df: pd.DataFrame, feature_col: str, target_col: str) -> LinearRegression:
    """
    Treina um modelo de Regressão Linear (sklearn) para prever salários
    a partir de anos de experiência.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame com os dados.
    feature_col : str
        Nome da coluna de feature (X).
    target_col : str
        Nome da coluna target (y).

    Returns
    -------
    LinearRegression
        Modelo treinado.
    """
    x = df[[feature_col]]
    y = df[target_col]

    model = LinearRegression()
    model.fit(x, y)

    coeficiente = model.coef_[0]
    intercepto = model.intercept_
    r2 = model.score(x, y)

    print("\n=== Modelo de Regressão Linear (sklearn) ===")
    print(f"Intercepto: {intercepto:.4f}")
    print(f"Coeficiente ({feature_col}): {coeficiente:.4f}")
    print(f"R²: {r2:.4f}")

    return model


def evaluate_model_rmse(model: LinearRegression, df: pd.DataFrame, feature_col: str, target_col: str) -> float:
    """
    Calcula o erro quadrático médio raiz (RMSE) do modelo de regressão.

    Parameters
    ----------
    model : LinearRegression
        Modelo de regressão treinado.
    df : pd.DataFrame
        DataFrame com os dados.
    feature_col : str
        Nome da coluna de feature (X).
    target_col : str
        Nome da coluna target (y).

    Returns
    -------
    float
        Valor do RMSE.
    """
    x = df[[feature_col]]
    y_true = df[target_col]

    y_pred = model.predict(x)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))

    print("\n=== Avaliação do Modelo ===")
    print(f"RMSE: {rmse:.4f}")

    return rmse


def predict_salaries(model: LinearRegression, experiences: List[float]) -> None:
    """
    Imprime previsões de salário para uma lista de anos de experiência.

    Parameters
    ----------
    model : LinearRegression
        Modelo treinado.
    experiences : List[float]
        Lista de valores de anos de experiência para previsão.
    """
    # sklearn espera um array 2D: [[x1], [x2], ...]
    x_new = [[exp] for exp in experiences]
    y_pred = model.predict(x_new)

    print("\n=== Previsões de Salário ===")
    for exp, salary in zip(experiences, y_pred):
        print(f"{exp:.0f} anos de experiência -> salário estimado: {salary:.2f}")


# ==========================
# Função Principal (orquestra tudo)
# ==========================

def main() -> None:
    # 1. Carregar dados
    df = load_salary_dataset(DATASET_PATH)

    # 2. Análise de outliers por IQR
    detect_outliers_iqr(df, COLUMNS_FOR_IQR, iqr_multiplier=1.5)

    # 3. Análise de influência na regressão (statsmodels)
    analyze_regression_influence(df, FEATURE_COLUMN, TARGET_COLUMN, threshold=2.0)

    # 4. Treinar modelo de Regressão Linear (sklearn)
    model = train_linear_regression(df, FEATURE_COLUMN, TARGET_COLUMN)

    # 5. Avaliar modelo com RMSE
    evaluate_model_rmse(model, df, FEATURE_COLUMN, TARGET_COLUMN)

    # 6. Fazer previsões para valores específicos de anos de experiência
    experiencias_para_prever = [3, 5, 10, 22]  # inclui o colaborador com 22 anos
    predict_salaries(model, experiencias_para_prever)


if __name__ == "__main__":
    main()



Coluna: YearsExperience
Limites IQR (1.5x): [-3.45, 14.55]
Outliers encontrados:
Nenhum

Coluna: Salary
Limites IQR (1.5x): [-9014.25, 166281.75]
Outliers encontrados:
Nenhum

=== Outliers de Regressão (statsmodels OLS) ===
Total de pontos com |resíduo studentizado| > 2.0: 1
    YearsExperience   Salary  resid_studentized  cooks_distance
19              6.1  93941.0           2.013697        0.074303

=== Modelo de Regressão Linear (sklearn) ===
Intercepto: 24848.2040
Coeficiente (YearsExperience): 9449.9623
R²: 0.9570

=== Avaliação do Modelo ===
RMSE: 5592.0436

=== Previsões de Salário ===
3 anos de experiência -> salário estimado: 53198.09
5 anos de experiência -> salário estimado: 72098.02
10 anos de experiência -> salário estimado: 119347.83
22 anos de experiência -> salário estimado: 232747.38


d:\Programas\Python\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
